# A case study: a backpack

**Every number in this notebook is invented.** Every amount and every factor was made up for this example and rounded. None comes from ecoinvent or any other licensed source, and none describes a real product, factory or country.

A brand sells a 20-litre daypack. It is sewn in a factory in Vietnam and sold in Europe. Its designers ask three questions:

1. What is the footprint of one backpack, and what makes it?
2. Would moving the factory to Portugal, or weaving the fabric from recycled PET, lower it? By more than the uncertainty?
3. What does a lighter fabric change?

This notebook answers them with headwater, step by step: open, validate, calculate, contributions, where, sensitivity, two scenarios, a Monte Carlo range, and a design change shown with `headwater diff`. Read [01-first-footprint](01-first-footprint.ipynb) and [02-writing-a-database](02-writing-a-database.ipynb) first: they explain each step on a smaller toy. The files are in [case-study/](case-study/README.md).

## Set up

This cell finds the case study's files next to this notebook, copies its project into a temporary folder, and pulls its release, as `headwater release pull` does for the demo. `HEADWATER_HOME` points into the same temporary folder, so your own cache is not touched.

`headwater(...)` is a small helper of this notebook, not part of headwater: it runs the `headwater` command next to this notebook's Python, prints what it prints, with the temporary folder shown as `<work>`, and stops if it fails.

In [1]:
import math
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

import headwater as hw

here = Path.cwd()
case_study = next(
    (folder for folder in (here / "case-study", here / "notebooks/case-study") if folder.is_dir()),
    None,
)
assert case_study, (
    "notebooks/case-study not found: run this notebook from a clone of the repository"
)

workdir = tempfile.TemporaryDirectory()
work = Path(workdir.name).resolve()
os.environ["HEADWATER_HOME"] = str(work / "home")
HEADWATER = shutil.which("headwater", path=str(Path(sys.executable).parent))


def headwater(*args, cwd=work):
    """Run the headwater command, print what it prints, and stop if it fails."""
    done = subprocess.run([HEADWATER, *map(str, args)], cwd=cwd, capture_output=True, text=True)
    print((done.stdout + done.stderr).replace(str(work), "<work>").rstrip())
    assert done.returncode == 0, f"exit code {done.returncode}"


project = work / "project"
shutil.copytree(case_study / "project", project)
shutil.copytree(case_study / "releases", work / "releases")
headwater("release", "pull", "releases/invented-background@9a8f8d0651d9")

Pulled release invented-background@9a8f8d0651d9 into <work>/home/releases


## The background

The backpack's suppliers are in a background of 20 invented processes, in the same two tables as any database. `processes.csv` lists them, with the place where each one runs: `VN` and `PT` for the two countries, `GLO` for the rest of the world.

In [2]:
print((case_study / "background/processes.csv").read_text())

id,name,location,kind
electricity-vn,electricity production,VN,ordinary_transforming_activity
electricity-pt,electricity production,PT,ordinary_transforming_activity
heat-natural-gas,heat production from natural gas,GLO,ordinary_transforming_activity
pet-granulate,PET granulate production,GLO,ordinary_transforming_activity
pet-granulate-recycled,"PET granulate production, recycled from bottles",GLO,ordinary_transforming_activity
polyester-fabric,polyester fabric production,VN,ordinary_transforming_activity
polyester-fabric-recycled,"polyester fabric production, recycled",VN,ordinary_transforming_activity
nylon-granulate,nylon 6.6 granulate production,GLO,ordinary_transforming_activity
nylon-webbing,nylon webbing production,VN,ordinary_transforming_activity
zinc-alloy,zinc alloy production,GLO,ordinary_transforming_activity
zip,zip production,VN,ordinary_transforming_activity
pp-granulate,polypropylene granulate production,GLO,ordinary_transforming_activity
buckle,plastic buckle product

Their lines are in `exchanges.csv`. This is the polyester fabric, woven in Vietnam. Per kg it buys 1.1 kg of PET granulate (10% is lost) and 10 kWh of Vietnam's electricity, and the mill's own boiler emits 1.9 kg of CO₂. The recycled fabric is the same, but it buys PET granulate made from bottles.

In [3]:
for row in (case_study / "background/exchanges.csv").read_text().splitlines():
    if row.startswith(("process,", "polyester-fabric,")):
        print(row)

process,line,role,flow,amount,unit,supplier,distribution,scale,minimum,maximum
polyester-fabric,e01,reference,invented:polyester-fabric,1,kg,,,,,
polyester-fabric,e02,input,invented:pet-granulate,1.1,kg,invented:pet-granulate,,,,
polyester-fabric,e03,input,invented:electricity,10,kWh,invented:electricity-vn,,,,
polyester-fabric,e04,emission,invented:co2,1.9,kg,,,,,


The method, `invented-climate`, has one indicator, climate change: 1 kg of CO₂ counts 1 kg CO₂-eq, methane 29.8 and nitrous oxide 273. The background and the method are frozen as the release `invented-background@9a8f8d0651d9`, which the set-up cell pulled.

**You type** `hw.load(name)`. It solves the score of every product of the background once. **You get** a `Background`, and `bg.score(id)` reads the score of one unit of any of its products.

In [4]:
bg = hw.load("invented-background@9a8f8d0651d9")
print(bg)
print()
for process_id, per in [
    ("invented:electricity-vn", "kWh"),
    ("invented:electricity-pt", "kWh"),
    ("invented:polyester-fabric", "kg"),
    ("invented:polyester-fabric-recycled", "kg"),
]:
    print(f"{process_id:36} {bg.score(process_id):9.5f} kg CO2-eq per {per}")

assert math.isclose(bg.score("invented:polyester-fabric"), 12.18368, rel_tol=1e-9)

Background(release invented-background@9a8f8d0651d9, method invented-climate 1, 1 indicator, 20 processes)

invented:electricity-vn                0.74470 kg CO2-eq per kWh
invented:electricity-pt                0.21490 kg CO2-eq per kWh
invented:polyester-fabric             12.18368 kg CO2-eq per kg
invented:polyester-fabric-recycled     9.85722 kg CO2-eq per kg


**By hand**, the fabric. Electricity in Vietnam is 0.7 kg CO₂ + 0.0015 kg methane × 29.8 = 0.7447 per kWh. Heat is 0.06 + 0.0002 × 29.8 = 0.06596 per MJ. PET granulate is 1.5 + 0.003 × 29.8 + 15 MJ × 0.06596 = 2.5788 per kg. So one kg of fabric is 1.1 × 2.5788 + 10 × 0.7447 + 1.9 = 2.83668 + 7.447 + 1.9 = 12.18368. Recycled granulate is 0.2 + 4 MJ × 0.06596 = 0.46384 per kg, so the recycled fabric is 9.857224.

## The backpack's database and study

The brand's own data is the database `project/backpack`, three processes. Each has its lines in `exchanges.csv`. The last four columns give some lines a distribution, for the Monte Carlo run below: the sea route, for example, is lognormal with a geometric standard deviation of 1.3 (`scale` is its natural log).

In [5]:
backpack = project / "backpack"
print((backpack / "processes.csv").read_text())
print((backpack / "exchanges.csv").read_text())

id,name,location,kind
backpack,"backpack, delivered and disposed of",GLO,ordinary_transforming_activity
sewing-vn,"backpack sewing, factory in Vietnam",VN,ordinary_transforming_activity
sewing-pt,"backpack sewing, factory in Portugal",PT,ordinary_transforming_activity

process,line,role,flow,amount,unit,supplier,distribution,scale,minimum,maximum
backpack,e01,reference,backpack:backpack,1,unit,,,,,
backpack,e02,input,backpack:sewn-backpack,1,unit,backpack:sewing-vn,,,,
backpack,e03,input,invented:sea-freight,9,tkm,invented:transport-ship,lognormal,0.2623643,,
backpack,e04,input,invented:road-freight,0.6,tkm,invented:transport-truck,,,,
backpack,e05,input,invented:cardboard,0.12,kg,invented:cardboard,,,,
backpack,e06,input,invented:ldpe-film,0.02,kg,invented:ldpe-film,,,,
backpack,e07,waste_to_treatment,invented:textile-waste,0.4,kg,invented:incineration,,,,
backpack,e08,waste_to_treatment,invented:textile-waste,0.4,kg,invented:landfill,,,,
sewing-vn,e01,reference,backpack:sewn-backpack

Read it from the top:

- `backpack` is one backpack, delivered to the warehouse in Europe and disposed of. It buys one sewn backpack from the factory in Vietnam (`e02`), 9 tkm by ship (1 kg over 9,000 km) and 0.6 tkm by lorry, a cardboard box and a film bag. At the end of its life, 0.4 kg is incinerated and 0.4 kg landfilled (`waste_to_treatment`).
- `sewing-vn` sews it in Vietnam: 0.6 kg of polyester fabric, 0.12 kg of nylon webbing, 2 zips, 0.08 kg of foam, 4 buckles and 1.2 kWh of Vietnam's electricity. 0.08 kg of scraps go to landfill.
- `sewing-pt` is the same factory in Portugal, on Portugal's electricity. Nothing buys it yet: a scenario will.

The question is a study, `project/backpack-footprint`, next to the database. It asks for one backpack, cradle to grave, and holds two scenarios.

In [6]:
footprint = project / "backpack-footprint"
for name in [
    "study.yaml",
    "scenarios/factory-in-portugal.yaml",
    "scenarios/recycled-polyester.yaml",
]:
    print(f"{name}:")
    print((footprint / name).read_text())

study.yaml:
name: backpack-footprint
goal: Footprint of one backpack, to compare a factory move and recycled polyester. Every number is invented.
functional_unit: 1 backpack, delivered to the warehouse in Europe and disposed of
products:
  - {process: backpack:backpack, amount: 1}
boundary: cradle-to-grave
methods: [invented-climate]
cut_off: {criteria: [mass], max_cumulative: 0.01}

scenarios/factory-in-portugal.yaml:
description: The backpack is sewn in Portugal. No sea freight, and 2,500 km by lorry instead of 600.
changes:
  - process: backpack:backpack
    line: e02
    supplier: backpack:sewing-pt
  - process: backpack:backpack
    line: e03
    amount: 0
  - process: backpack:backpack
    line: e04
    amount: 2.5

scenarios/recycled-polyester.yaml:
description: The fabric is woven from recycled PET instead of virgin PET.
changes:
  - process: backpack:sewing-vn
    line: e02
    supplier: invented:polyester-fabric-recycled



## Open and validate

**You type** `hw.open(folder)`, which reads the study, and `study.validate(bg)`, which checks it and the database it reaches against the background: every supplier found, every unit known, every flow counted by the method, the scenarios applicable. **You get** a report of errors and warnings. Here it is empty.

In [7]:
study = hw.open(footprint)
print(study)
report = study.validate(bg)
print(report)
assert not report.problems

Study(backpack-footprint: 1 product, methods invented-climate, scenarios factory-in-portugal recycled-polyester)
Report(no problems)


## Calculate

**By hand**, from the scores per unit of the background, each line is its amount × the score of what it buys:

| Line | Amount × score per unit | kg CO₂-eq |
|---|---|---:|
| fabric | 0.6 kg × 12.18368 | 7.310208 |
| webbing | 0.12 kg × 10.7829 | 1.293948 |
| zips | 2 × 0.166744 | 0.333488 |
| foam | 0.08 kg × 3.59152 | 0.287322 |
| buckles | 4 × 0.052214 | 0.208855 |
| electricity in Vietnam | 1.2 kWh × 0.7447 | 0.893640 |
| scraps to landfill | 0.08 kg × 0.0598 | 0.004784 |
| **sewing in Vietnam** | | **10.332245** |
| ship and lorry | 9 tkm × 0.01 + 0.6 tkm × 0.12 | 0.162000 |
| cardboard and film | 0.12 kg × 0.79576 + 0.02 kg × 2.0596 | 0.136683 |
| end of life | 0.4 kg × 2.0 + 0.4 kg × 0.0598 | 0.823920 |
| **one backpack** | | **11.454848** |

**You type** `study.calc(bg)`. **You get** one `Result` per product of the study, here one. The cell checks it against 11.454848.

In [8]:
[result] = study.calc(bg)
print(result)
assert math.isclose(result.score, 11.454848, rel_tol=1e-9)

Result(backpack:backpack = 11.4548 kg CO2-eq, indicator climate-change, method invented-climate 1)


## What is it made of?

`result.contributions()` follows what the backpack buys, level by level. Each node is a process, with its score, its share of the backpack, and its direct emissions.

In [9]:
tree = result.contributions()
print(tree)

[sewing] = [child for child in tree.children if child.process_id == "backpack:sewing-vn"]
assert math.isclose(sewing.score, 10.332245, rel_tol=1e-6)

1 unit of backpack, delivered and disposed of: 11.45485 (100.0%), direct 0.00000
  1 unit of backpack sewing, factory in Vietnam: 10.33224 (90.2%), direct 0.00000
    0.6 kg of polyester fabric production: 7.31021 (63.8%), direct 1.14000
      6 kWh of electricity production: 4.46820 (39.0%), direct 4.46820
      0.66 kg of PET granulate production: 1.70201 (14.9%), direct 1.04900
    0.12 kg of nylon webbing production: 1.29395 (11.3%), direct 0.00000
      0.126 kg of nylon 6.6 granulate production: 0.75776 (6.6%), direct 0.54999
      0.72 kWh of electricity production: 0.53618 (4.7%), direct 0.53618
    1.2 kWh of electricity production: 0.89364 (7.8%), direct 0.89364
    rest: 0.83445 (7.3%)
  0.4 kg of incineration of textile waste: 0.80000 (7.0%), direct 0.80000
  rest: 0.32260 (2.8%)


**It means.** Sewing in Vietnam is 90% of the backpack, because it buys the materials. The fabric alone is 64%, and most of the fabric is the electricity that weaves it. Transport is small: the ship is 9 tkm × 0.01 = 0.09. End of life is 7%, almost all of it the incinerator.

## Who emits it, and where?

`result.where()` answers another question: which processes emit, wherever they are upstream, and in which place. This is where a reduction plan starts: an emission happens somewhere.

In [10]:
print(result.where())

1 unit of backpack, delivered and disposed of: 11.45485
By process:
  8.2 kWh of electricity production (VN): 6.10654 (53.3%)
  0.6 kg of polyester fabric production (VN): 1.14000 (10.0%)
  16.25 MJ of heat production from natural gas (GLO): 1.07185 (9.4%)
  0.66 kg of PET granulate production (GLO): 1.04900 (9.2%)
  0.4 kg of incineration of textile waste (GLO): 0.80000 (7.0%)
  0.15 kg of nylon 6.6 granulate production (GLO): 0.65475 (5.7%)
  0.08 kg of polyurethane foam production (GLO): 0.22400 (2.0%)
  9 tkm of transport, container ship (GLO): 0.09000 (0.8%)
  0.06 kg of polypropylene granulate production (GLO): 0.07200 (0.6%)
  0.6 tkm of transport, lorry (GLO): 0.07200 (0.6%)
  rest: 0.17470 (1.5%), 4 processes
By location:
  VN: 7.24654 (63.3%), 2 processes
  GLO: 4.20831 (36.7%), 12 processes


**It means.** Vietnam's power plants emit more than half of the footprint: 8.2 kWh run for one backpack, 6 of them for the fabric. With the fabric mills' boilers, 63% of the footprint is emitted in Vietnam. The brand buys only 1.2 kWh of it directly, so the tree of `contributions` spreads the rest over the materials.

## Which lines matter most?

`result.sensitivity()` ranks every line someone could change, in the database and in the background, by its **elasticity**: by how many % the footprint changes when the line's amount goes up by 1%.

In [11]:
print(f"{'process':28} {'line':4} {'role':9} {'amount':>9} {'elasticity':>10}")
for line in result.sensitivity(top=6):
    amount = f"{line.amount:g} {line.unit}"
    label = f"{line.process_id:28} {line.line_id:4} {line.role.value:9}"
    print(f"{label} {amount:>9} {line.elasticity:10.4f}")

process                      line role         amount elasticity
backpack:backpack            e02  input        1 unit     0.9020
backpack:sewing-vn           e02  input        0.6 kg     0.6382
invented:electricity-vn      e02  emission     0.7 kg     0.5011
invented:polyester-fabric    e03  input        10 kWh     0.3901
invented:polyester-fabric    e02  input        1.1 kg     0.1486
backpack:sewing-vn           e03  input       0.12 kg     0.1130


**It means.** The brand's own lines that matter most are the factory itself (`backpack` e02) and the fabric it buys (`sewing-vn` e02): 1% less fabric is 0.64% less footprint. In the background, the grid of Vietnam (`electricity-vn` e02) and the fabric mill's electricity come next. Those point at the two scenarios: another grid, another fabric.

## Two scenarios

A scenario is a file of the study with a list of changes, shown above. `factory-in-portugal` buys the sewn backpack from `sewing-pt`, drops the ship and drives 2,500 km by lorry. `recycled-polyester` buys the fabric from the recycled mill. The database files do not change.

**By hand.** In Portugal the sewing's 1.2 kWh costs 0.2149 instead of 0.7447 per kWh: −0.63576. No ship: −0.09. The lorry goes from 0.6 to 2.5 tkm: +0.228. In all −0.49776, so 10.957088. With recycled PET the 0.6 kg of fabric costs 9.857224 instead of 12.18368 per kg: −1.3958736, so 10.0589744.

**You type** `study.calc(bg, scenario=name)`.

In [12]:
[portugal] = study.calc(bg, scenario="factory-in-portugal")
[recycled] = study.calc(bg, scenario="recycled-polyester")
for name, scored in [
    ("as it is", result),
    ("factory in Portugal", portugal),
    ("recycled polyester", recycled),
]:
    print(f"{name:20} {scored.score:.5f} kg CO2-eq")

assert math.isclose(portugal.score, 10.957088, rel_tol=1e-9)
assert math.isclose(recycled.score, 10.0589744, rel_tol=1e-9)

as it is             11.45485 kg CO2-eq
factory in Portugal  10.95709 kg CO2-eq
recycled polyester   10.05897 kg CO2-eq


`where` shows what the move does to the places: Portugal appears, with the sewing's electricity, and Vietnam keeps the fabric.

In [13]:
print(portugal.where())

1 unit of backpack, delivered and disposed of: 10.95709
By process:
  7 kWh of electricity production (VN): 5.21290 (47.6%)
  0.6 kg of polyester fabric production (VN): 1.14000 (10.4%)
  16.25 MJ of heat production from natural gas (GLO): 1.07185 (9.8%)
  0.66 kg of PET granulate production (GLO): 1.04900 (9.6%)
  0.4 kg of incineration of textile waste (GLO): 0.80000 (7.3%)
  0.15 kg of nylon 6.6 granulate production (GLO): 0.65475 (6.0%)
  2.5 tkm of transport, lorry (GLO): 0.30000 (2.7%)
  1.2 kWh of electricity production (PT): 0.25788 (2.4%)
  0.08 kg of polyurethane foam production (GLO): 0.22400 (2.0%)
  0.06 kg of polypropylene granulate production (GLO): 0.07200 (0.7%)
  rest: 0.17470 (1.6%), 4 processes
By location:
  VN: 6.35290 (58.0%), 2 processes
  GLO: 4.34631 (39.7%), 11 processes
  PT: 0.25788 (2.4%), 1 process


## How sure are we?

Five lines carry a distribution: the CO₂ of each grid (geometric standard deviation 1.1), of virgin PET granulate (1.2), the nitrous oxide of nylon (1.5), and the sea route (1.3). **You type** `hw.monte_carlo(bg, items, iterations=..., seed=...)`. It draws those lines 1,000 times and scores each item in each draw. The three items here are the backpack as it is and in each scenario. They share the same draws, so in each draw they are compared on the same grid, the same PET and the same route.

The cell prints the median and the range that holds 95% of the draws, rounded.

In [14]:
items = [
    (study, "backpack:backpack"),
    (study, "backpack:backpack", "factory-in-portugal"),
    (study, "backpack:backpack", "recycled-polyester"),
]
mc = hw.monte_carlo(bg, items, iterations=1000, seed=0)


def show(label, spread):
    low, high = spread.percentiles[2.5], spread.percentiles[97.5]
    print(f"{label:34} median {spread.median:6.2f}, 95% of draws {low:6.2f} to {high:6.2f}")


for item, label in zip(
    mc.items, ["as it is", "factory in Portugal", "recycled polyester"], strict=True
):
    show(label, mc.range(item))
print()
show("as it is − factory in Portugal", mc.difference(mc.items[0], mc.items[1]))
show("as it is − recycled polyester", mc.difference(mc.items[0], mc.items[2]))

base_draws, portugal_draws, recycled_draws = mc.samples[:, 0, :]
print()
print(f"Portugal is lower in {(portugal_draws < base_draws).mean():.0%} of the draws")
print(f"recycled polyester is lower in {(recycled_draws < base_draws).mean():.0%} of the draws")
assert (portugal_draws < base_draws).all() and (recycled_draws < base_draws).all()

as it is                           median  11.48, 95% of draws  10.45 to  12.89
factory in Portugal                median  10.98, 95% of draws  10.07 to  12.22
recycled polyester                 median  10.07, 95% of draws   9.10 to  11.39

as it is − factory in Portugal     median   0.50, 95% of draws   0.34 to   0.69
as it is − recycled polyester      median   1.40, 95% of draws   1.10 to   1.78

Portugal is lower in 100% of the draws
recycled polyester is lower in 100% of the draws


**It means.** Alone, each score has a wide range, about 10% either side, and the ranges of the three overlap. Read alone, they would say the options cannot be told apart. But the options share most of their supply chain, and so most of their uncertainty. Drawn together, both are lower in every draw: recycled polyester by about 1.1 to 1.8 kg, Portugal by about 0.3 to 0.7 kg. That is why headwater draws every item of a run from the same draws.

## A design change, as a diff

The fabric was the brand's own line that mattered most. The designer picks a lighter fabric: 0.5 kg instead of 0.6 per backpack. The project is text, kept in git, so the change is a commit like any change of code, and `headwater diff` shows what it does to the footprint and why.

This cell makes a throwaway git repository of the project and commits it. The author and the dates are fixed, so the commit hash it prints is the same on every run.

In [15]:
identity = {
    "GIT_AUTHOR_NAME": "Notebook",
    "GIT_AUTHOR_EMAIL": "notebook@example.com",
    "GIT_AUTHOR_DATE": "2026-11-01T12:00:00Z",
    "GIT_COMMITTER_NAME": "Notebook",
    "GIT_COMMITTER_EMAIL": "notebook@example.com",
    "GIT_COMMITTER_DATE": "2026-11-01T12:00:00Z",
}
for command in (
    ["init", "-q", "-b", "main"],
    ["add", "."],
    ["-c", "commit.gpgsign=false", "commit", "-q", "-m", "The backpack"],
):
    subprocess.run(["git", *command], cwd=project, env={**os.environ, **identity}, check=True)

Now change line e02 of `sewing-vn` from 0.6 to 0.5 kg, without committing, and ask `headwater diff` what that does. With one ref, it compares that ref with the files as they are now.

**By hand**: 0.1 kg less fabric × 12.18368 = 1.218368 less, so 11.454848 − 1.218368 = 10.23648.

In [16]:
lines = backpack / "exchanges.csv"
lines.write_text(
    lines.read_text().replace(
        "sewing-vn,e02,input,invented:polyester-fabric,0.6,",
        "sewing-vn,e02,input,invented:polyester-fabric,0.5,",
    )
)
headwater("diff", "main", "backpack-footprint", cwd=project)

[lighter] = hw.open(footprint).calc(bg)
assert math.isclose(lighter.score, 10.23648, rel_tol=1e-9)

Study backpack-footprint: main (f88544efe506) → working tree
Background: invented-background@9a8f8d0651d9, method invented-climate 1

1 × backpack, delivered and disposed of | backpack (backpack:backpack)
  Climate change: 11.4548 → 10.2365 kg CO2-eq, change -1.21837 (-10.6%)

Lines that differ, ranked by their effect on Climate change (kg CO2-eq):
  -1.21837  backpack:sewing-vn line e02 (polyester fabric): amount 0.6 → 0.5 kg


**It means.** The diff scores the study at `main` and as it is now, and names the one line that changed as the cause. On a pull request, CI posts the same report as a comment ([workflows.md](../docs/workflows.md#ci)), so a reviewer sees the effect of a change before it is merged.

## What you did

- Read an invented background and the brand's own database and study, all as text files.
- Validated and calculated one backpack, and checked 11.454848 kg CO₂-eq by hand.
- Saw what it is made of (`contributions`), where it is emitted (`where`), and which lines matter most (`sensitivity`).
- Scored two scenarios without touching the database, and saw with a Monte Carlo run that both are lower in every draw, although their ranges overlap.
- Made a design change in git and read its effect with `headwater diff`.

Next: [04-working-with-ecoinvent](04-working-with-ecoinvent.ipynb) does this on a real background, ecoinvent 3.12, which needs a licence. [docs/workflows.md](../docs/workflows.md) has every command.